# Tutorial 01: Web Scraping from BCRP

This notebook demonstrates how to scrape Weekly Reports (PDFs) from the Central Reserve Bank of Peru (BCRP) website using Selenium.

## What You'll Learn

1. How to set up Selenium WebDriver
2. Navigate the BCRP website programmatically
3. Download PDFs with retry logic
4. Implement rate limiting to mimic human behavior
5. Track downloaded files to avoid duplicates

## Prerequisites

- Python 3.10+
- Chrome/Firefox/Edge browser installed
- `peru_gdp_rtd` package installed

## Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from peru_gdp_rtd.config import get_settings
from peru_gdp_rtd.scrapers import pdf_downloader, init_driver
from peru_gdp_rtd.scrapers.utils import random_wait

# Load configuration
settings = get_settings('../config/config.yaml')

print(f"Project: {settings.project.name} v{settings.project.version}")
print(f"Browser: {settings.scraper.browser}")
print(f"Max downloads: {settings.scraper.max_downloads}")

## Step 1: Initialize WebDriver

The WebDriver controls the browser programmatically.

In [ ]:
# Initialize driver (headless=False to see the browser)
driver = init_driver(
    browser=settings.scraper.browser,
    headless=False,  # Set to True to run in background
    page_load_timeout=30
)

print("WebDriver initialized successfully!")
print(f"Browser: {driver.name}")

## Step 2: Navigate to BCRP Website

The BCRP publishes Weekly Reports at: https://www.bcrp.gob.pe/publicaciones/nota-semanal.html

In [ ]:
# URL to BCRP Weekly Reports page
bcrp_url = "https://www.bcrp.gob.pe/publicaciones/nota-semanal.html"

# Navigate to the page
driver.get(bcrp_url)

# Wait for page to load
import time
time.sleep(3)

print(f"Page title: {driver.title}")
print(f"Current URL: {driver.current_url}")

## Step 3: Find PDF Links

Use Selenium to locate PDF download links on the page.

In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Wait for PDF links to be present
wait = WebDriverWait(driver, 10)
pdf_links = wait.until(
    EC.presence_of_all_elements_located((By.XPATH, "//a[contains(@href, '.pdf')]"))
)

print(f"Found {len(pdf_links)} PDF links")

# Display first 5 links
for i, link in enumerate(pdf_links[:5], 1):
    href = link.get_attribute('href')
    text = link.text.strip()
    print(f"{i}. {text[:50]:50s} -> {href}")

## Step 4: Download a Sample PDF

Download one PDF to test the process.

In [ ]:
import requests
from pathlib import Path

# Get first PDF link
if pdf_links:
    sample_link = pdf_links[0]
    pdf_url = sample_link.get_attribute('href')
    pdf_name = Path(pdf_url).name
    
    print(f"Downloading: {pdf_name}")
    print(f"URL: {pdf_url}")
    
    # Download PDF
    response = requests.get(pdf_url, timeout=30)
    
    if response.status_code == 200:
        # Save to new_weekly_reports folder
        output_path = Path('../new_weekly_reports') / pdf_name
        output_path.parent.mkdir(exist_ok=True)
        
        with open(output_path, 'wb') as f:
            f.write(response.content)
        
        print(f"✓ Downloaded successfully: {output_path}")
        print(f"  File size: {len(response.content) / 1024:.1f} KB")
    else:
        print(f"✗ Download failed: HTTP {response.status_code}")

## Step 5: Rate Limiting

To avoid being blocked by the server, we implement rate limiting.

In [ ]:
import random
import time

def random_wait(min_sec: float = 1.0, max_sec: float = 3.0):
    """Wait a random amount of time to mimic human behavior."""
    wait_time = random.uniform(min_sec, max_sec)
    print(f"Waiting {wait_time:.2f} seconds...")
    time.sleep(wait_time)

# Example usage
print("Download 1")
random_wait(1.0, 3.0)
print("Download 2")
random_wait(1.0, 3.0)
print("Download 3")

## Step 6: Full Scraping Pipeline

Use the built-in `pdf_downloader` function to download all PDFs.

In [ ]:
# Close the test driver first
driver.quit()

# Run full download pipeline
# WARNING: This will download ~60 PDFs and may take 15-30 minutes

# Uncomment to run:
# pdf_downloader(
#     browser=settings.scraper.browser,
#     headless=settings.scraper.headless,
#     max_downloads=5,  # Limit to 5 for testing
#     rate_limit=(1.0, 3.0),
# )

print("To run the full download, uncomment the code above.")
print("Or use the command line: python scripts/update_rtd.py --steps 1")

## Step 7: Verify Downloaded Files

In [ ]:
from pathlib import Path
import os

# List downloaded PDFs
download_folder = Path('../new_weekly_reports')

if download_folder.exists():
    pdf_files = list(download_folder.glob('**/*.pdf'))
    
    print(f"Total PDFs downloaded: {len(pdf_files)}")
    print("\nRecent downloads:")
    
    # Sort by modification time (most recent first)
    pdf_files_sorted = sorted(pdf_files, key=lambda x: x.stat().st_mtime, reverse=True)
    
    for i, pdf in enumerate(pdf_files_sorted[:10], 1):
        size_kb = pdf.stat().st_size / 1024
        print(f"{i:2d}. {pdf.name:40s} ({size_kb:6.1f} KB)")
else:
    print("No downloads yet. Run the downloader first.")

## Key Takeaways

1. **Selenium WebDriver**: Automates browser interaction
2. **Rate Limiting**: Prevents server overload and blocking
3. **Error Handling**: Retry logic for failed downloads
4. **Progress Tracking**: tqdm for visual feedback
5. **Idempotency**: Skip already downloaded files

## Next Steps

- **Tutorial 02**: PDF Processing - Extract GDP tables from downloaded PDFs
- **Tutorial 03**: Data Cleaning - Standardize and normalize the data

## Additional Resources

- [Selenium Documentation](https://selenium-python.readthedocs.io/)
- [BCRP Weekly Reports](https://www.bcrp.gob.pe/publicaciones/nota-semanal.html)
- [Main Pipeline Documentation](../README.md)